# Combined-Probe Constraints (AGD Phase 3a)

**The falsifiable version of the extra-dimensional / dark-energy thread.**

This notebook does *hypothesis testing*, kept strictly decoupled from the pipeline's *agnostic anomaly discovery* (biasing the anomaly hunt toward a preferred explanation manufactures confirmations). None of these datasets 'detect a dimension'. A braneworld/DGP-type model earns or loses credibility by **where (w0, wa, γ, S8) land** relative to GR+ΛCDM.

Every number is transcribed from its source paper in `src/analysis/constraints.py` (arXiv IDs attached). *No narrative without a number.* The heavy relations live in unit-tested modules (`src/analysis/cosmology.py`); this notebook orchestrates and renders the verdict.

Staged, in order of constraining power:
1. ΛCDM sanity check
2. DESI DR2 BAO w0waCDM
3. + CMB (Planck 2018) priors
4. + SNe (DES-SN5YR)
5. + growth axis (KiDS-1000, DES Y3 — S8/γ)
6. **Verdict cell** — GR+ΛCDM vs braneworld/DGP

In [1]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


ROOT = _find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

repo root: /home/user/astrophysics-data-engineering-codex


In [2]:
from src.analysis import constraints as C
from src.analysis import cosmology as cosmo
print('fiducial: Planck18  H0 =', cosmo.PLANCK18_H0, ' Om0 =', cosmo.PLANCK18_OM0)

fiducial: Planck18  H0 = 67.36  Om0 = 0.3153


## Stage 1 — ΛCDM sanity check
The baseline every dynamical fit is compared against. w(z) ≡ −1.

In [3]:
lcdm = cosmo.build_cosmology(w0=-1.0, wa=0.0)
print('model:', type(lcdm).__name__)
print('D_A(z=1)      =', round(lcdm.angular_diameter_distance(1.0).value, 1), 'Mpc')
print('D_C(z=1)      =', round(lcdm.comoving_distance(1.0).value, 1), 'Mpc')
print('w(z) sanity   :', [cosmo.cpl_w(z, -1.0, 0.0) for z in (0, 1, 3)])
print('growth f(0)   =', round(cosmo.growth_rate(0.0, gamma=C.GAMMA_GR), 4),
      '(GR γ =', C.GAMMA_GR, ')')

model: FlatLambdaCDM
D_A(z=1)      = 1701.3 Mpc
D_C(z=1)      = 3402.7 Mpc
w(z) sanity   : [-1.0, -1.0, -1.0]
growth f(0)   = 0.53 (GR γ = 0.55 )


## Stage 2 — DESI DR2 BAO, w0waCDM
arXiv:2503.14738. Favoured quadrant is **w0 > −1, wa < 0**; preference over ΛCDM grows as SNe are added (2.8–4.2σ).

In [4]:
hdr = f"{'combination':22s} {'w0':>16s} {'wa':>16s} {'Om':>8s} {'H0':>7s} {'σ':>5s}"
print(hdr); print('-' * len(hdr))
for rec in C.DESI_DR2_W0WA.values():
    print(f'{rec.label:22s} {str(rec.w0):>16s} {str(rec.wa):>16s} '
          f'{rec.omega_m.value:>8.4f} {rec.h0.value:>7.2f} {rec.sigma_vs_lcdm:>4.1f}σ')

combination                          w0               wa       Om      H0     σ
-------------------------------------------------------------------------------
DESI+CMB                   -0.43 ± 0.21       -1.7 ± 0.6   0.3520   63.70  3.1σ
DESI+CMB+Pantheon+       -0.853 ± 0.057     -0.54 ± 0.22   0.3117   67.62  2.8σ
DESI+CMB+Union3          -0.678 ± 0.092 -1.03 (+0.3/-0.29)   0.3273   65.98  3.8σ
DESI+CMB+DESY5           -0.752 ± 0.057 -0.86 (+0.23/-0.2)   0.3191   66.74  4.2σ


## Stage 3 — + CMB (Planck 2018) priors
arXiv:1807.06209, Table 2 (TT,TE,EE+lowE+lensing). The high-z anchor that breaks the H0–r_d and w0–wa degeneracies; the S8 baseline the growth tension is measured against.

In [5]:
for k, m in C.PLANCK18.items():
    print(f'{k:12s} = {m}')
print('source:', C.PLANCK18_SOURCE, '(arXiv:' + C.PLANCK18_ARXIV + ')')

omega_m      = 0.3153 ± 0.0073
h0           = 67.36 ± 0.54
sigma8       = 0.8111 ± 0.006
s8           = 0.832 ± 0.013
n_s          = 0.9649 ± 0.0042
omega_b_h2   = 0.02237 ± 0.00015
source: Planck 2018 VI, Table 2 (TT,TE,EE+lowE+lensing) (arXiv:1807.06209)


## Stage 4 — + SNe (DES-SN5YR)
arXiv:2401.02929. The third combined-probe leg; already folded into the DESI+CMB+DESY5 row above, where the ΛCDM preference peaks at 4.2σ.

In [6]:
print('DES-SN5YR flat-ΛCDM  Om =', C.DES_SN5YR_OMEGA_M)
print('source:', C.DES_SN5YR_SOURCE, '(arXiv:' + C.DES_SN5YR_ARXIV + ')')

DES-SN5YR flat-ΛCDM  Om = 0.352 ± 0.017
source: DES-SN5YR (DES Collaboration 2024) (arXiv:2401.02929)


## Stage 5 — + growth axis (KiDS-1000, DES Y3)
The axis that most directly separates modified gravity from ΛCDM **today**, pre-Euclid-DR1. Both Stage-III lensing S8 values sit *low* relative to Planck — the S8 tension, the direction several modified-gravity/braneworld models predict.

In [7]:
planck_s8 = C.PLANCK18['s8']
print(f"Planck 2018   S8 = {planck_s8}")
for g in C.STAGE3_GROWTH.values():
    t = cosmo.tension_sigma(planck_s8, g.s8)
    pct = cosmo.distance_from_lcdm_percent(g.s8, lcdm_value=planck_s8.value)
    print(f'{g.label:16s} S8 = {str(g.s8):>16s}  '
          f'{t:.1f}σ below Planck  ({pct:+.1f}%)   [arXiv:{g.arxiv}]')

Planck 2018   S8 = 0.832 ± 0.013
KiDS-1000 3x2pt  S8 = 0.766 (+0.02/-0.014)  2.8σ below Planck  (-7.9%)   [arXiv:2007.15632]
DES Y3 3x2pt     S8 =    0.776 ± 0.017  2.6σ below Planck  (-6.7%)   [arXiv:2105.13549]


## Stage 6 — Verdict
Model credibility is earned or lost here. No narrative without a number.

In [8]:
best = C.DESI_DR2_W0WA['DESI+CMB+DESY5']
n_w0 = (best.w0.value - (-1.0)) / best.w0.error_toward(-1.0)
kids_t = cosmo.tension_sigma(C.PLANCK18['s8'], C.STAGE3_GROWTH['KiDS-1000'].s8)
des_t = cosmo.tension_sigma(C.PLANCK18['s8'], C.STAGE3_GROWTH['DES-Y3'].s8)

verdict = {
    'strongest_combo': best.label,
    'w0': best.w0.value, 'wa': best.wa.value,
    'w0_above_-1_sigma': round(n_w0, 1),
    'DE_preference_over_LCDM_sigma': best.sigma_vs_lcdm,
    'quadrant': 'w0>-1, wa<0' if (best.w0.value > -1 and best.wa.value < 0) else 'other',
    'S8_tension_KiDS_sigma': round(kids_t, 1),
    'S8_tension_DESY3_sigma': round(des_t, 1),
    'gamma_GR': C.GAMMA_GR, 'gamma_DGP_selfaccel': C.GAMMA_DGP,
}
for k, v in verdict.items():
    print(f'{k:32s}: {v}')

print('\n--- plain-language verdict (built from the numbers above) ---')
print(
    f"Combined probes prefer dynamical dark energy over ΛCDM at up to "
    f"{best.sigma_vs_lcdm:.1f}σ ({best.label}); the best fit sits in the "
    f"{verdict['quadrant']} quadrant, with w0 {n_w0:.1f}σ above −1. Growth (S8) "
    f"runs ~{kids_t:.1f}σ (KiDS) / {des_t:.1f}σ (DES Y3) low vs Planck-ΛCDM. "
    f"This is the (w0>-1, wa<0) + low-S8 corner that thawing-quintessence and "
    f"some braneworld models occupy — while self-accelerating DGP (γ≈{C.GAMMA_DGP}) "
    f"is separately excluded by growth. Verdict: a constraint, not a detection. "
    f"A braneworld/DGP model earns credibility only by predicting this whole "
    f"corner at once without violating γ and lab torsion-balance bounds."
)

strongest_combo                 : DESI+CMB+DESY5
w0                              : -0.752
wa                              : -0.86
w0_above_-1_sigma               : 4.4
DE_preference_over_LCDM_sigma   : 4.2
quadrant                        : w0>-1, wa<0
S8_tension_KiDS_sigma           : 2.8
S8_tension_DESY3_sigma          : 2.6
gamma_GR                        : 0.55
gamma_DGP_selfaccel             : 0.68

--- plain-language verdict (built from the numbers above) ---
Combined probes prefer dynamical dark energy over ΛCDM at up to 4.2σ (DESI+CMB+DESY5); the best fit sits in the w0>-1, wa<0 quadrant, with w0 4.4σ above −1. Growth (S8) runs ~2.8σ (KiDS) / 2.6σ (DES Y3) low vs Planck-ΛCDM. This is the (w0>-1, wa<0) + low-S8 corner that thawing-quintessence and some braneworld models occupy — while self-accelerating DGP (γ≈0.68) is separately excluded by growth. Verdict: a constraint, not a detection. A braneworld/DGP model earns credibility only by predicting this whole corner at once without

### November 2026 checkpoint — DR1-Foundation
When consortium weak-lensing likelihoods/chains publish (~1900 deg²), fold the Euclid growth constraint into Stage 5 and re-run this cell. The verdict is re-runnable against DR1F with no code change — only the numbers in `constraints.py` update.